In [ ]:
import pandas as pd
import numpy as np
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns
import os


print("[코호트 생성]")

# 데이터 로드
try:
    patients = pd.read_csv('icu_patients.csv')
    admissions = pd.read_csv('icu_admissions.csv')
    icustays = pd.read_csv('icu_icustays.csv')
except FileNotFoundError:
    print("[오류] csv 파일을 업로드해주세요.")
    raise

# 병합
merged = pd.merge(patients, admissions, on='subject_id', how='inner')
cohort = pd.merge(merged, icustays, on=['subject_id', 'hadm_id'], how='inner')

# 시간 변환
cohort['intime'] = pd.to_datetime(cohort['intime'])
cohort['outtime'] = pd.to_datetime(cohort['outtime'])
cohort['admittime'] = pd.to_datetime(cohort['admittime'])
cohort['dischtime'] = pd.to_datetime(cohort['dischtime'])

# [1차 필터링]

# 1. 2011 ~ 2022 전체 포함
target_years = ['2011 - 2013', '2014 - 2016', '2017 - 2019', '2020 - 2022']
cohort = cohort[cohort['anchor_year_group'].isin(target_years)].copy()

# 2. ELECTIVE 제외
cohort = cohort[cohort['admission_type'] != 'ELECTIVE']

# 성인 환자만 선택 (18세 이상)
cohort = cohort[cohort['anchor_age'] >= 18]
print(f"성인 환자(18세 이상)만 선택: {len(cohort):,} 명")

# 3. 음수 LOS 제외
cohort['icu_los_days'] = (cohort['outtime'] - cohort['intime']).dt.total_seconds() / 86400
cohort = cohort[cohort['icu_los_days'] > 0]

# 4. 첫 번째 입실만 유지
cohort = cohort.sort_values(['subject_id', 'intime'])
cohort = cohort.drop_duplicates(subset=['subject_id'], keep='first')

# 5. 윈도우 변수
cohort['win_start'] = cohort['intime']
cohort['win_end'] = cohort['intime'] + pd.Timedelta(hours=48)

# 약물 필터링 이전 상태

print("-" * 60)
print(f"약물 필터링 전(Pre-Drug) 코호트 크기")
print(f"   - Shape: {cohort.shape}")
print(f"   - 환자 수: {len(cohort):,} 명")
print("-" * 60)


# [2차 필터링] DB 연동 및 약물 투여 확인

# Google Drive
from google.colab import drive
drive.mount('/content/drive')
# DB 경로
db_path = '/content/drive/MyDrive/MIMIC4-hosp-icu.db'

try:
    conn = sqlite3.connect('temp_cohort_check.db')
    # 필요한 컬럼만 업로드
    cohort[['stay_id', 'intime']].to_sql('cohort_for_drug', conn, if_exists='replace', index=False)
    conn.execute(f"ATTACH DATABASE '{db_path}' AS remote_db")

    # 약물 필터링 쿼리
    drug_sql = """
    SELECT DISTINCT t.stay_id
    FROM cohort_for_drug t
    JOIN remote_db.inputevents i ON t.stay_id = i.stay_id
    WHERE i.endtime > t.intime
    """

    valid_drug_stays = pd.read_sql_query(drug_sql, conn)
    conn.close()

    # 필터링 적용
    final_cohort = cohort[cohort['stay_id'].isin(valid_drug_stays['stay_id'])].copy()

    # 약물 필터링 이후 상태

    print("-" * 60)
    print(f"약물 필터링 후(Post-Drug) 코호트 크기")
    print(f"   - Shape: {final_cohort.shape}")
    print(f"   - 환자 수: {len(final_cohort):,} 명")
    print("-" * 60)

except Exception as e:
    print(f"[오류] DB 연결 실패: {e}")
    final_cohort = cohort

# 추가 파생 변수

final_cohort['log_los'] = np.log1p(final_cohort['icu_los_days'])
final_cohort['pre_icu_los'] = (final_cohort['intime'] - final_cohort['admittime']).dt.total_seconds() / 86400
final_cohort['pre_icu_los'] = final_cohort['pre_icu_los'].clip(lower=0)

# 정적 변수 결측치 처리
cols_fill = ['marital_status', 'insurance', 'language']
for c in cols_fill:
    if c in final_cohort.columns:
        final_cohort[c] = final_cohort[c].fillna('UNKNOWN')

if 'race' in final_cohort.columns:
    final_cohort['race_simplified'] = final_cohort['race'].apply(
        lambda x: x if x in ['WHITE', 'BLACK/AFRICAN AMERICAN', 'ASIAN', 'HISPANIC/LATINO'] else 'OTHER'
    )

# 연도별 약물 커버리지 확인

print("\n연도별 약물 데이터 보유 비율")
count_pre = cohort['anchor_year_group'].value_counts().sort_index()
count_post = final_cohort['anchor_year_group'].value_counts().sort_index()

stats_df = pd.DataFrame({'Total': count_pre, 'Drug_Filtered': count_post})
stats_df['Percentage'] = (stats_df['Drug_Filtered'] / stats_df['Total']) * 100
print(stats_df)

In [ ]:
import pandas as pd
import numpy as np
import sqlite3
import os

print("   [Final Pipeline] Lab 피처 추가 및 최종 데이터셋 생성   ")


from google.colab import drive
drive.mount('/content/drive')

db_path = '/content/drive/MyDrive/MIMIC4-hosp-icu.db'


# Base Cohort
input_file = 'base_and_vital_final.csv'
try:
    base_df = pd.read_csv(input_file)
    print(f"\n[1] Base Cohort 로드 완료 ('{input_file}')")
    print(f"   - 전체 환자 수: {len(base_df):,} 명")

    # 시간 컬럼 변환
    base_df['intime'] = pd.to_datetime(base_df['intime'])

except FileNotFoundError:
    print(f"[오류] '{input_file}' 파일을 Colab에 업로드해주세요.")
    raise

# Lab 데이터 추출

try:
    conn = sqlite3.connect('local_lab_final.db')

    # DB JOIN을 위한 임시 테이블 생성
    cohort_for_sql = base_df[['stay_id', 'subject_id', 'intime']].copy()
    cohort_for_sql.to_sql('temp_cohort', conn, if_exists='replace', index=False, dtype={'intime': 'DATETIME'})

    # DB 연결
    conn.execute(f"ATTACH DATABASE '{db_path}' AS remote_db")

    # 추출할 Lab 항목 (12개: Lactate 포함, Albumin/Bilirubin 제거)
    lab_item_ids = (
        50912, 51006, 51301, 51265, # Cr, BUN, WBC, Plt
        50813, 50882, 50971, 50983, 50868, # Lactate, Bicarb, K, Na, AnionGap
        50931, 51237, 50820  # Glu, INR, pH
    )

    # 48시간 내 데이터 조회 쿼리
    lab_sql = f"""
    SELECT
        t.stay_id,
        l.itemid,
        l.valuenum
    FROM remote_db.labevents l
    JOIN temp_cohort t ON l.subject_id = t.subject_id
    WHERE l.itemid IN {lab_item_ids}
      AND l.charttime >= t.intime
      AND l.charttime < DATETIME(t.intime, '+48 hours')
      AND l.valuenum IS NOT NULL
    """

    lab_df = pd.read_sql_query(lab_sql, conn)
    conn.close()
    print(f"   - 추출된 Lab 기록 수: {len(lab_df):,} 건")

except Exception as e:
    print(f"[오류] DB 연결 실패: {e}")
    raise

# 이름 매핑
item_map = {
    50912: 'Creatinine', 51006: 'BUN', 51301: 'WBC', 51265: 'Platelet',
    50813: 'Lactate', 50882: 'Bicarbonate', 50971: 'Potassium',
    50983: 'Sodium', 50868: 'AnionGap',
    50931: 'Glucose', 51237: 'INR', 50820: 'pH'
}
lab_df['item_name'] = lab_df['itemid'].map(item_map)

# 이상치 처리 루프
for item in item_map.values():
    mask = (lab_df['item_name'] == item)
    data = lab_df.loc[mask, 'valuenum']

    if len(data) > 0:
        lower = data.quantile(0.01)
        upper = data.quantile(0.99)
        lab_df.loc[mask, 'valuenum'] = data.clip(lower=lower, upper=upper)


# 피처 생성 (Aggregation)
print("\n[4] 피처 집계 (Mean, Min, Max, Std)...")

lab_agg = lab_df.pivot_table(
    index='stay_id',
    columns='item_name',
    values='valuenum',
    aggfunc=['mean', 'min', 'max', 'std']
)

# 컬럼명 정리: (mean, Glucose) -> Glucose_mean
lab_agg.columns = [f"{col[1]}_{col[0]}" for col in lab_agg.columns]
lab_agg = lab_agg.reset_index()

print(f"   - 생성된 Lab 피처 수: {len(lab_agg.columns) - 1} 개")

# 최종 병합 및 결측치 대치 (Imputation)

print("\n[5] 병합 및 결측치 대치 (Median Imputation)...")

# 1. 병합
final_dataset = pd.merge(base_df, lab_agg, on='stay_id', how='left')

# 2. 결측치 처리
# 전략: Std는 0으로, 나머지는 중앙값(Median)으로 채움

# (A) Std 컬럼 -> 0.0
std_cols = [c for c in final_dataset.columns if c.endswith('_std')]
final_dataset[std_cols] = final_dataset[std_cols].fillna(0.0)

# (B) Mean, Min, Max 컬럼 -> Median
stat_cols = [c for c in final_dataset.columns if c.endswith('_mean') or c.endswith('_min') or c.endswith('_max')]
# 중앙값 계산
fill_values = final_dataset[stat_cols].median()
final_dataset[stat_cols] = final_dataset[stat_cols].fillna(fill_values)

# (C) Vital 쪽 결측치도 혹시 모르니 한 번 더 처리 (동일 로직)
vital_stat_cols = [c for c in final_dataset.columns if ('HR_' in c or 'MAP_' in c) and ('_mean' in c or '_min' in c)]
if len(vital_stat_cols) > 0:
    vital_fill = final_dataset[vital_stat_cols].median()
    final_dataset[vital_stat_cols] = final_dataset[vital_stat_cols].fillna(vital_fill)

print(f"   - 결측치 처리 완료.")
print(f"   - Lactate_mean 남은 결측치: {final_dataset['Lactate_mean'].isnull().sum()} 개 (0이어야 함)")

# 저장
output_file = 'FINAL.csv'
final_dataset.to_csv(output_file, index=False)

print("\n==========================================================")
print(f"최종 파일: '{output_file}'")
print(f"   - Final Shape: {final_dataset.shape}")
print("==========================================================")